# トークン数を数える (LLM のコンテキスト長・料金の見積もり)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/06_subword/count_tokens.ipynb)

NLP クックブック 06-10 ／ サブワードとトークナイザ

テキストが LLM の何トークンになるかを数えたい。日本語と英語でトークン数がどれくらい違うかも知りたい。

In [1]:
!pip install -q tiktoken

In [2]:
import tiktoken

enc = tiktoken.get_encoding("o200k_base")  # GPT-4o 系の語彙
for text in ["Natural language processing is fun.", "自然言語処理は楽しい。", "吾輩は猫である。名前はまだ無い。"]:
    ids = enc.encode(text)
    print(len(text), "文字", len(ids), "トークン", [enc.decode([i]) for i in ids])

35 文字 6 トークン ['Natural', ' language', ' processing', ' is', ' fun', '.']
11 文字 9 トークン ['自然', '言', '語', '処', '理', 'は', '楽', 'しい', '。']
16 文字 14 トークン ['吾', '�', '�', 'は', '猫', 'で', 'ある', '。', '名前', 'は', 'まだ', '無', 'い', '。']


In [3]:
# オープンなモデルは Hugging Face のトークナイザで数える
from transformers import AutoTokenizer

text = "自然言語処理は楽しい。吾輩は猫である。名前はまだ無い。"
for name in ["Qwen/Qwen2.5-0.5B-Instruct", "xlm-roberta-base", "google/byt5-small"]:
    tok = AutoTokenizer.from_pretrained(name)
    print(f"{name:30} {len(tok(text, add_special_tokens=False)['input_ids'])} トークン")

Qwen/Qwen2.5-0.5B-Instruct     21 トークン
xlm-roberta-base               17 トークン
google/byt5-small              81 トークン


- 1 文字が複数トークンに分かれると、トークン単体では表示できず `�` になる (上の「輩」)。
- 日本語は「1 文字 ≒ 1 トークン前後」、英語は「1 単語 ≒ 1.3 トークン」が大まかな目安。モデルによって大きく違うので実測する。
- API 型のモデルは各社が公式のトークン数カウント API を提供している。厳密な見積もりにはそちらを使う。